In [36]:
import pandas as pd

### Loading Data

In [37]:
df = pd.read_csv("IMDB Dataset.csv")

In [38]:
df.head()
df.isnull().sum()

review       0
sentiment    0
dtype: int64

# Data Preprocessing

### 1- Removing Duplicate Values

In [39]:
df.drop_duplicates(inplace = True)

In [40]:
df.shape

(49582, 2)

### 2- Converting to lowercase

In [41]:
df["review"] = df["review"].str.lower()

### 3- Removing URLs

In [42]:
import re

In [43]:
def remove_urls(text):
    text = re.sub(r"http\S+","",text) # (pattern , repl , string)
    return text


df["review"] = df["review"].apply(remove_urls)

### 4- Remove Punctuations

In [44]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [45]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 5- Remove html tags

In [46]:
def remove_html(text):
    text = re.sub(r"<.*","",text)
    return text

df["review"] = df["review"].apply(remove_html)

In [47]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 6- Making tokens and Removing stopwords

In [48]:
import nltk

nltk.download("punkt") # Official tokenizer in nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hacke\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hacke\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hacke\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [50]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word,"")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [51]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 7- Stemming (PorterStemming)

In [52]:
from nltk.stem import PorterStemmer

In [53]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_tokens = ps.stem(token)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words)


df["review"] = df["review"].apply(stemming)

### 8- Encoding

In [58]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]

### 9- Vectorization

In [59]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000) # Only need 5000 important features 

X = tf.fit_transform(df["review"])

# Train Test Split

In [65]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42 
)

# DataSet & DataLoaders

In [66]:
import torch
from torch.utils.data import TensorDataset , DataLoader

In [67]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [74]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values.copy()).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values.copy()).float()
)

In [76]:
train_loader = DataLoader(train_set , batch_size = 64 , shuffle = True)
test_loader = DataLoader(test_set , batch_size = 64 , shuffle = True)

# RNN Model

In [78]:
import torch.nn as nn
import torch.optim as optim

In [79]:
class RNN(nn.Module):
    def __init__(self , input_size , hidden_size = 128 , num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN Model
        self.rnn = nn.RNN(input_size , hidden_size , num_layers , batch_first = True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        
        out , _  = self.rnn(x)
        out = self.fc(out[:,-1,:])
        return out

In [80]:
input_size = X_train.shape[1]
model = RNN(input_size)

In [81]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [83]:
epochs = 10 

for epoch in range(epochs):
    model.train()


    for Xb , yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # Add Singleton direction
        outputs = model(Xb)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => Probability 
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

    print(f"epoch : {epoch+1} loss : {loss.item()}")

epoch : 1 loss : 0.23838600516319275
epoch : 2 loss : 0.1576070636510849
epoch : 3 loss : 0.38566359877586365
epoch : 4 loss : 0.43013036251068115
epoch : 5 loss : 0.22446733713150024
epoch : 6 loss : 0.4451921284198761
epoch : 7 loss : 0.1874079406261444
epoch : 8 loss : 0.14898644387722015
epoch : 9 loss : 0.24241496622562408
epoch : 10 loss : 0.2799527049064636


### Evaluation

In [84]:
model.eval()

with torch.no_grad():
    correct_val = 0
    tot_val = 0

    for Xb , yb in test_loader:
        Xb = Xb.unsqueeze(1)
        outputs = model(Xb)

        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_val += yb.size(0)
        correct_val += (predicted == yb).sum().item()

    print(f"Accuracy : {(correct_val / tot_val )* 100}")

Accuracy : 85.55006554401533
